In [1]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import requests
import spotipy
import lyricsgenius
import faiss
import os
import itertools
import random
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from spotipy.oauth2 import SpotifyClientCredentials
from sentence_transformers import SentenceTransformer

Loading the pre-trained BGE-M3 model into memory

In [3]:
model = SentenceTransformer("BAAI/bge-m3")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading in our processed tracks and genres/tags dataset

In [4]:
final_df = pd.read_pickle("../data/processed/final_merged_data.pkl")

In [5]:
print("Fetching genres from Every Noise at Once...")
response = requests.get("https://everynoise.com/")
soup = BeautifulSoup(response.text, "html.parser")

every_noise_genres = {
    div.text.strip().lower().replace('»', '').strip() 
    for div in soup.find_all('div', class_='genre')
}
print(f"Loaded {len(every_noise_genres)} unique genres.")

normalized_noise_genres = {
    g.replace('-', '').replace(' ', '').replace('_', '').replace('/', '').replace("'", ""): g 
    for g in every_noise_genres
}

custom_genres = {
    'heavymetal': 'heavy metal',
    'punkrock': 'punk rock',
    'hardrock': 'hard rock',
    'britishpunk': 'british punk',
    'rocknroll': 'rock and roll',
    'rnb': 'r&b'
}
normalized_noise_genres.update(custom_genres)

def split_genres_normalized(tag_list):
    if not isinstance(tag_list, list):
        return [], []
    
    genres, descriptors = [], []
    
    for tag in tag_list:
        lower_tag = tag.lower()
        norm_tag = lower_tag.replace('-', '').replace(' ', '').replace('_', '').replace('/', '').replace("'", "")
        
        if norm_tag in normalized_noise_genres:
            genres.append(normalized_noise_genres[norm_tag]) 
        else:
            descriptors.append(lower_tag)
            
    return list(set(genres)), descriptors

Fetching genres from Every Noise at Once...
Loaded 6291 unique genres.


Generating a user taste profile based on their top tracks from Last.fm and our FAISS index, to combine it into a centroid vector

In [6]:
index = faiss.read_index("../data/processed/lyric_embeddings.index")

load_dotenv("../.env")
SPOTIPY_CLIENT_ID = os.getenv("SPOTIPY_CLIENT_ID")
SPOTIPY_CLIENT_SECRET = os.getenv("SPOTIPY_CLIENT_SECRET")
GENIUS_ACCESS_TOKEN = os.getenv("GENIUS_ACCESS_TOKEN")
LASTFM_API_KEY = os.getenv("LASTFM_API_KEY")
LASTFM_USERNAME = os.getenv("LASTFM_USERNAME")

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=SPOTIPY_CLIENT_ID,
    client_secret=SPOTIPY_CLIENT_SECRET
))

genius = lyricsgenius.Genius(
    GENIUS_ACCESS_TOKEN, 
    timeout=15,
    retries=3
)

def fetch_lastfm_tags(artist, track, api_key):
    url = "https://ws.audioscrobbler.com/2.0/"
    
    params = {
        'method': 'track.gettoptags',
        'artist': artist,
        'track': track,
        'api_key': api_key,
        'format': 'json'
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        tags = [tag['name'] for tag in data.get('toptags', {}).get('tag', [])]
        if tags:
            return tags
            
    params = {
        'method': 'artist.gettoptags',
        'artist': artist,
        'api_key': api_key,
        'format': 'json'
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        return [tag['name'] for tag in data.get('toptags', {}).get('tag', [])]
        
    return []

def get_track_vector_and_meta(artist_name, track_title, model):    
    local_match = final_df[
        (final_df['artist'].str.lower() == artist_name.lower()) & 
        (final_df['song'].str.lower() == track_title.lower())
    ]
    
    if not local_match.empty:
        row_idx = local_match.index[0]
        vector = index.reconstruct(int(row_idx))
        genres = local_match.iloc[0]['genres']
        return vector, genres
    
    print(f"'{track_title}' not in local FAISS index. Fetching live...")
    
    raw_tags = fetch_lastfm_tags(artist_name, track_title, LASTFM_API_KEY)
    
    clean_genres, _ = split_genres_normalized(raw_tags) 
    
    song = genius.search_song(track_title, artist_name)
    if not song or not song.lyrics:
        return None, None
        
    vector = model.encode(song.lyrics)
    
    return vector, clean_genres

def fetch_lastfm_user_profile(username, api_key, limit=20):
    url = "https://ws.audioscrobbler.com/2.0/"
    params = {
        'method': 'user.gettoptracks',
        'user': username,
        'api_key': api_key,
        'format': 'json',
        'limit': limit
    }
    
    response = requests.get(url, params=params)
    if response.status_code != 200:
        print("Error fetching Last.fm profile.")
        return []
        
    data = response.json()
    tracks = []
    
    for item in data.get('toptracks', {}).get('track', []):
        track_name = item['name']
        artist_name = item['artist']['name']
        tracks.append({'artist': artist_name, 'song': track_name})
        
    return tracks

def build_user_centroid(user_tracks, model):
    vectors = []
    
    for track in user_tracks:
        vector, _ = get_track_vector_and_meta(track['artist'], track['song'], model)
        if vector is not None:
            vectors.append(vector)
            
    if not vectors:
        raise ValueError("Could not resolve any vectors for the user profile.")
        
    centroid = np.mean(np.array(vectors), axis=0).astype('float32')
    return centroid

FAISS search function to the closest matching tracks to the user centroid in the database

In [7]:
def generate_recommendations(centroid, user_tracks, top_k=5, mode="similar"):
    search_depth = 50 if mode == "similar" else 150 
    
    distances, indices = index.search(np.array([centroid]), search_depth)
    
    recommendations = []
    user_artists = {track['artist'].lower() for track in user_tracks}
    
    for i, idx in enumerate(indices[0]):
        row = final_df.iloc[idx]
        artist = row['artist']
        
        if artist.lower() in user_artists:
            continue
            
        if mode == "discovery" and i < 20:
            continue
            
        recommendations.append({
            'artist': artist,
            'song': row['song'],
            'genres': row['genres'],
            'similarity_score': round(float(distances[0][i]), 4)
        })
        
        if len(recommendations) >= top_k:
            break
            
    return pd.DataFrame(recommendations)

print(f"Fetching profile for user...")
my_tracks = fetch_lastfm_user_profile(LASTFM_USERNAME, LASTFM_API_KEY, limit=15)

print(f"Encoding tracks and building centroid vector...")
my_centroid = build_user_centroid(my_tracks, model)

print("\n--- SIMILAR RECOMMENDATIONS ---")
similar_df = generate_recommendations(my_centroid, my_tracks, top_k=5, mode="similar")
display(similar_df)

print("\n--- DISCOVERY RECOMMENDATIONS ---")
discovery_df = generate_recommendations(my_centroid, my_tracks, top_k=5, mode="discovery")
display(discovery_df)

Fetching profile for user...
Encoding tracks and building centroid vector...
'115' not in local FAISS index. Fetching live...
'Death Blooms' not in local FAISS index. Fetching live...
'Slow Chemical' not in local FAISS index. Fetching live...
'The One' not in local FAISS index. Fetching live...
'Save Theme' not in local FAISS index. Fetching live...
'Forget to Remember - From Saw II - Original Soundtrack' not in local FAISS index. Fetching live...
'Dull Boy' not in local FAISS index. Fetching live...
'You Make Me Sick' not in local FAISS index. Fetching live...
'Beauty of Annihilation' not in local FAISS index. Fetching live...
'Fall Into Sleep' not in local FAISS index. Fetching live...
'Mercy, Severity' not in local FAISS index. Fetching live...
'Under and Over It' not in local FAISS index. Fetching live...
'Hero' not in local FAISS index. Fetching live...

--- SIMILAR RECOMMENDATIONS ---


,artist,song,genres,similarity_score
0,Kirk Franklin,Hide Me,"[gospel, soul]",0.2809
1,Kirk Franklin,How It Used To Be,"[worship, gospel]",0.3039
2,Korn,Starting Over,"[nu metal, rock, heavy metal, metal, experimen...",0.3070
3,Mariah Carey,Outside,"[pop, r&b]",0.3182
4,Chris Brown,Fallen Angel,"[rap, hip hop, fallen angel, r&b]",0.3216



--- DISCOVERY RECOMMENDATIONS ---


,artist,song,genres,similarity_score
0,Richard Marx,Hold On To The Nights,"[soft rock, rock, easy listening, pop rock, si...",0.3409
1,Zoegirl,Stop Right There,[pop],0.3420
2,Maroon 5,Better That We Break,"[soft rock, rock, soul, alternative rock, pop ...",0.3425
3,Kelly Clarkson,Fading,"[rock, pop rock, acoustic rock]",0.3426
4,Mariah Carey,I Don't Wanna Cry,"[soul, quiet storm, singer-songwriter, r&b, rh...",0.3429


Evaluating the model<br>
Metrics: Precision@K (Historical Accuracy), Centroid Distance (Novelty), and Intra-List Diversity

In [8]:
import itertools
import random

def evaluate_metrics(recommendations_df, test_tracks):
    if recommendations_df.empty:
        return 0.0, 0.0, 0.0
        
    #Precision@K
    test_artists = {t['artist'].lower() for t in test_tracks}
    hits = recommendations_df['artist'].str.lower().isin(test_artists).sum()
    precision = hits / len(recommendations_df)
    
    #Centroid Distance
    avg_centroid_dist = recommendations_df['similarity_score'].mean()
    
    #Intra-List Diversity
    vectors = []
    for _, row in recommendations_df.iterrows():
        idx = final_df[(final_df['artist'] == row['artist']) & (final_df['song'] == row['song'])].index[0]
        vectors.append(index.reconstruct(int(idx)))
        
    ild = 0.0
    if len(vectors) > 1:
        distances = [np.linalg.norm(v1 - v2) for v1, v2 in itertools.combinations(vectors, 2)]
        ild = np.mean(distances)
        
    return round(precision, 4), round(avg_centroid_dist, 4), round(ild, 4)

print("Splitting profile into Train/Test sets for model evaluation...")
random.shuffle(my_tracks)
train_tracks = my_tracks[:10]
test_tracks = my_tracks[10:]

print("Building evaluation centroid...")
eval_centroid = build_user_centroid(train_tracks, model)

print("Generating recommendation sets...")
sim_eval_df = generate_recommendations(eval_centroid, train_tracks, top_k=5, mode="similar")
disc_eval_df = generate_recommendations(eval_centroid, train_tracks, top_k=5, mode="discovery")

sim_p, sim_nov, sim_ild = evaluate_metrics(sim_eval_df, test_tracks)
disc_p, disc_nov, disc_ild = evaluate_metrics(disc_eval_df, test_tracks)

print("\n=== EVALUATION RESULTS ===")
print(f"SIMILAR MODE   -> Precision@{len(sim_eval_df)}: {sim_p} | Novelty: {sim_nov} | Diversity (ILD): {sim_ild}")
print(f"DISCOVERY MODE -> Precision@{len(disc_eval_df)}: {disc_p} | Novelty: {disc_nov} | Diversity (ILD): {disc_ild}")

Splitting profile into Train/Test sets for model evaluation...
Building evaluation centroid...
'Mercy, Severity' not in local FAISS index. Fetching live...
'You Make Me Sick' not in local FAISS index. Fetching live...
'Beauty of Annihilation' not in local FAISS index. Fetching live...
'Forget to Remember - From Saw II - Original Soundtrack' not in local FAISS index. Fetching live...
'115' not in local FAISS index. Fetching live...
'Death Blooms' not in local FAISS index. Fetching live...
'Save Theme' not in local FAISS index. Fetching live...
'Hero' not in local FAISS index. Fetching live...
Generating recommendation sets...

=== EVALUATION RESULTS ===
SIMILAR MODE   -> Precision@5: 0.0 | Novelty: 0.3224 | Diversity (ILD): 0.730400025844574
DISCOVERY MODE -> Precision@5: 0.0 | Novelty: 0.358 | Diversity (ILD): 0.7505000233650208
